### Maximilian Kempa LAB | Programming Foundation Exercises

This simulator creates and updates different smart home sensors with random readings.
Each reading is checked against security, safety, and comfort rules depending on the current system mode.
If an abnormal condition is detected, the system displays an alert and logs the event.

In [116]:
import random
import datetime
import time

In [ ]:
def create_sensor(id, location, type):
    return {
        "id": id,
        "location": location,
        "type": type,
        "current_value": None
    }


def create_alert(sensor_id, alert_type, message):
    return {
        "sensor_id": sensor_id,
        "alert_type": alert_type,
        "message": message
    }


sensor_data = [
    create_sensor(1, "Living Room", "Motion"),
    create_sensor(2, "Front Door", "Door"),
    create_sensor(3, "Kitchen", "Temperature"),
    create_sensor(4, "Bedroom", "Smoke")
]

alerts = []

print(sensor_data)

In [117]:
class Sensor:
    def __init__(self, id, location, type):
        self.id = id
        self.location = location
        self.type = type
        self.current_value = None
        self.open_since = None

    def update_value(self, value):
        self.current_value = value

    def read(self):
        return self.current_value

    def isAbnormal(self):
        if self.type == "Temperature":
            return self.current_value < 35 or self.current_value > 95

        elif self.type == "Smoke":
            return self.current_value

        return False

    def reset(self):
        self.current_value = None
        self.open_since = None

In [118]:
motion_sensor = Sensor(id=1, location="Living Room", type="Motion")
door_sensor = Sensor(id=2, location="Front Door", type="Door")
temperature_sensor = Sensor(id=3, location="Kitchen", type="Temperature")
smoke_sensor = Sensor(id=4, location="Bedroom", type="Smoke")

sensors = [motion_sensor, door_sensor, temperature_sensor, smoke_sensor]

In [119]:
def generate_reading(sensor):

    if sensor.type == "Temperature":
        return random.randint(30, 100)

    elif sensor.type == "Motion":
        return random.random() < 0.3

    elif sensor.type == "Door":
        return random.random() < 0.3

    elif sensor.type == "Smoke":
        return random.random() < 0.05

In [120]:
system_mode = "AWAY"

In [121]:
event_log = []

def log_event(message):
    current_time = datetime.datetime.now().strftime("%H:%M:%S")
    log_message = f"[{current_time}] {message}"
    event_log.append(log_message)
    print(f"[LOG] {log_message}")

In [122]:
def trigger_alert(message):
    print(f"[ALERT!] 🚨 {message}")
    log_event("Sending notification to homeowner...")

In [123]:
def sensor_update(sensor, mode):

    if sensor.type == "Smoke":
        if sensor.current_value:
            print(f"[READING] {sensor.location} Smoke: DETECTED")
            trigger_alert(f"CRITICAL: Smoke detected in {sensor.location}!")
        else:
            print(f"[READING] {sensor.location} Smoke: CLEAR")

    elif sensor.type == "Temperature":
        if sensor.current_value < 35 or sensor.current_value > 95:
            print(f"[READING] {sensor.location} Temperature: {sensor.current_value}°F")
            trigger_alert(f"WARNING: Abnormal temperature in {sensor.location}!")

        elif mode == "HOME" and (sensor.current_value < 65 or sensor.current_value > 75):
            print(f"[READING] {sensor.location} Temperature: {sensor.current_value}°F")
            print(f"[NOTICE] Comfort: Temperature in {sensor.location} is outside the comfort range!")

        else:
            print(f"[READING] {sensor.location} Temperature: {sensor.current_value}°F (Normal)")

    elif sensor.type == "Motion":
        if sensor.current_value:
            print(f"[READING] {sensor.location} Motion: Activity detected")

            if mode == "AWAY":
                trigger_alert(f"HIGH SECURITY: Motion detected in {sensor.location} while system is in {mode} mode!")

            elif mode == "SLEEP":
                trigger_alert(f"MEDIUM SECURITY: Motion detected in {sensor.location} while system is in {mode} mode!")

        else:
            print(f"[READING] {sensor.location} Motion: No activity")

    elif sensor.type == "Door":
        if sensor.current_value:
            print(f"[READING] {sensor.location}: OPENED")

            if sensor.open_since is None:
                sensor.open_since = datetime.datetime.now()

            elif datetime.datetime.now() - sensor.open_since > datetime.timedelta(minutes=5):
                print(f"[NOTICE] {sensor.location} has been open for more than 5 minutes!")

            if mode == "AWAY":
                trigger_alert(f"HIGH SECURITY: {sensor.location} opened while system is in {mode} mode!")

            elif mode == "SLEEP":
                trigger_alert(f"MEDIUM SECURITY: {sensor.location} opened while system is in {mode} mode!")

        else:
            print(f"[READING] {sensor.location}: CLOSED")
            sensor.open_since = None

def check_multiple_security_alerts(motion_sensor, door_sensor, mode):
    if motion_sensor.current_value and door_sensor.current_value and mode == "AWAY":
        trigger_alert("CRITICAL SECURITY: Multiple sensors triggered - possible break-in!")

In [124]:
system_mode = "AWAY"
motion_sensor.update_value(True)
door_sensor.update_value(True)

In [ ]:
print("=== HomeGuard Security System ===")

while True:
#for _ in range(2):
    current_time = datetime.datetime.now().strftime("%H:%M:%S")

    print(f"\nTime: {current_time}")
    print(f"Mode: {system_mode}")
    print()

    for sensor in sensors:
        sensor.update_value(generate_reading(sensor))

    for sensor in sensors:
        sensor_update(sensor, system_mode)

    check_multiple_security_alerts(
        motion_sensor,
        door_sensor,
        system_mode
    )

    time.sleep(60)
    #time.sleep(2)

=== HomeGuard Security System ===

Time: 11:34:03
Mode: AWAY

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 56°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 11:34:05
Mode: AWAY

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 77°F (Normal)
[READING] Bedroom Smoke: CLEAR


In [126]:
# Security Test
system_mode = "AWAY"

motion_sensor.update_value(True)
door_sensor.update_value(True)
temperature_sensor.update_value(68)
smoke_sensor.update_value(False)

for sensor in sensors:
    sensor_update(sensor, system_mode)

check_multiple_security_alerts(motion_sensor, door_sensor, system_mode)

[READING] Living Room Motion: Activity detected
[ALERT!] 🚨 HIGH SECURITY: Motion detected in Living Room while system is in AWAY mode!
[LOG] [11:34:08] Sending notification to homeowner...
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH SECURITY: Front Door opened while system is in AWAY mode!
[LOG] [11:34:08] Sending notification to homeowner...
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR
[ALERT!] 🚨 CRITICAL SECURITY: Multiple sensors triggered - possible break-in!
[LOG] [11:34:08] Sending notification to homeowner...


In [127]:
# Safety Test
system_mode = "HOME"

motion_sensor.update_value(False)
door_sensor.update_value(False)
temperature_sensor.update_value(30)
smoke_sensor.update_value(True)

for sensor in sensors:
    sensor_update(sensor, system_mode)

check_multiple_security_alerts(motion_sensor, door_sensor, system_mode)

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 30°F
[ALERT!] 🚨 WARNING: Abnormal temperature in Kitchen!
[LOG] [11:34:08] Sending notification to homeowner...
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🚨 CRITICAL: Smoke detected in Bedroom!
[LOG] [11:34:08] Sending notification to homeowner...


In [128]:
# Comfort Test
system_mode = "HOME"

motion_sensor.update_value(False)
door_sensor.update_value(False)
temperature_sensor.update_value(60)
smoke_sensor.update_value(False)

for sensor in sensors:
    sensor_update(sensor, system_mode)

check_multiple_security_alerts(motion_sensor, door_sensor, system_mode)

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 60°F
[NOTICE] Comfort: Temperature in Kitchen is outside the comfort range!
[READING] Bedroom Smoke: CLEAR
